# Individual Exercise: Mini Data Quality Audit

Homework notebook for Week 2.


## Instructions

Perform a mini data quality audit and submit this completed notebook.


In [3]:
from pathlib import Path
import pandas as pd
import numpy as np
repo_root = Path.cwd().resolve().parents[1]
data_path = repo_root / 'data' / 'raw' / 'week2_customer_transactions_messy.csv'
df = pd.read_csv(data_path)
print('Loaded:', data_path)
print('Shape:', df.shape)
df.head()


Loaded: /Users/z.yang/playground/srh_data-management/data/raw/week2_customer_transactions_messy.csv
Shape: (11, 9)


,transaction_id,customer_id,transaction_date,amount,currency,payment_method,status,region,last_updated
0,T0001,C100,2026-01-05,120.50,EUR,card,completed,DE,2026-01-05
1,T0002,C101,2026/01/06,0.00,EUR,CARD,completed,de,2026-01-20
2,T0003,C102,06-01-2026,-35.00,USD,bank_transfer,completed,US,2026-01-07
3,T0004,NaN,2026-01-07,250.00,EUR,card,pending,FR,2026-01-08
4,T0005,C104,2026-01-07,89.99,EURO,cash,completed,DE,2026-01-09


## Required tasks

1. describe the dataset briefly
2. identify issues by dimension
3. compute at least 3 KPIs
4. define at least 3 validation rules
5. create a short audit summary
6. recommend cleaning steps for next chapter (do not implement full pipelines)


## Task 1 - Dataset description

### Your answer

This dataset contains customer transaction records, where each row represents a single transaction. It includes transaction and customer identifiers, transaction date, amount, currency, payment method, status, region, and the last_updated field.

A possible business use case is tracking and managing customer transactions. The data can be used to record payments, monitor transaction status, and support basic reporting of financial activities.

## Task 2 - Issues by dimension


In [10]:
issue_table = pd.DataFrame([
    ['Missing customer_id','Completeness','Impacts customer analytics'],
    ['Missing amount', 'Completeness', 'Prevents accurate financial calculations'],
    ['Duplicate transaction_id','Uniqueness','May double count revenue'],
    ['Invalid transaction_date', 'Validity', 'Affects time-based analysis and reporting'],
    ['Negative or zero amount', 'Validity', 'Indicates incorrect or suspicious transactions'],
    ['Inconsistent payment_method (e.g., card vs CARD)', 'Consistency', 'Causes issues in grouping and aggregation'],
    ['Inconsistent currency (EUR vs EURO)', 'Consistency', 'Leads to incorrect financial aggregation'],
    ['Inconsistent region format (DE vs de)', 'Consistency', 'Affects regional analysis'],
    ['last_updated before transaction_date', 'Integrity', 'Violates logical relationship between timestamps'],
], columns=['Issue','Dimension','Impact'])
issue_table


,Issue,Dimension,Impact
0,Missing customer_id,Completeness,Impacts customer analytics
1,Missing amount,Completeness,Prevents accurate financial calculations
2,Duplicate transaction_id,Uniqueness,May double count revenue
3,Invalid transaction_date,Validity,Affects time-based analysis and reporting
4,Negative or zero amount,Validity,Indicates incorrect or suspicious transactions
5,"Inconsistent payment_method (e.g., card vs CARD)",Consistency,Causes issues in grouping and aggregation
6,Inconsistent currency (EUR vs EURO),Consistency,Leads to incorrect financial aggregation
7,Inconsistent region format (DE vs de),Consistency,Affects regional analysis
8,last_updated before transaction_date,Integrity,Violates logical relationship between timestamps


## Task 3 - KPI calculations


In [5]:
kpis={}
kpis['Completeness Rate']=1-(df.isna().sum().sum()/(df.shape[0]*df.shape[1]))
kpis['Duplication Rate']=df.duplicated(subset=['transaction_id']).mean()
amount=pd.to_numeric(df['amount'], errors='coerce')
date_ok=pd.to_datetime(df['transaction_date'], errors='coerce', format='mixed').notna()
kpis['Error Rate']=(amount.isna() | (amount<0) | ~date_ok).mean()
pd.DataFrame({'KPI':list(kpis.keys()), 'Value':list(kpis.values())})


,KPI,Value
0,Completeness Rate,0.959596
1,Duplication Rate,0.090909
2,Error Rate,0.272727


### Your KPI interpretation

The completeness rate (0.96) indicates that most fields are filled, with only a small proportion of missing values. This suggests that data collection is generally reliable, although some gaps still exist.

The duplication rate (0.09) shows that there are duplicate transaction records in the dataset. This could lead to double counting and should be addressed before analysis.

The error rate (0.27) is relatively high, meaning that a significant portion of the records contain invalid values (e.g., incorrect dates or invalid transaction amounts). This indicates notable data quality issues that may affect the accuracy of any analysis.

Overall, while the dataset is mostly complete, the presence of duplicates and a high error rate suggests that data cleaning is necessary before further use.

## Task 4 - Validation rules


In [8]:
rules={
'transaction_id_required': df['transaction_id'].isna() | (df['transaction_id'].astype(str).str.strip()==''),
'amount_non_negative': pd.to_numeric(df['amount'], errors='coerce')<0,
'transaction_date_valid': pd.to_datetime(df['transaction_date'], errors='coerce', format='mixed').isna(),
}

# Invalid payment method (not in predefined valid list or inconsistent formatting)
valid_payment_methods = ['card', 'bank_transfer', 'cash']
rules['payment_method_invalid'] = ~df['payment_method'].astype(str).str.lower().isin(valid_payment_methods)

# Invalid currency values (non-standard currency codes)
valid_currencies = ['EUR', 'USD']
rules['currency_invalid'] = ~df['currency'].isin(valid_currencies)

# Inconsistent region format (not uppercase)
rules['region_inconsistent'] = df['region'].astype(str) != df['region'].astype(str).str.upper()

# Missing customer ID (required field)
rules['customer_id_missing'] = df['customer_id'].isna()

# Logical inconsistency: last_updated earlier than transaction_date
transaction_date = pd.to_datetime(df['transaction_date'], errors='coerce', format='mixed')
last_updated = pd.to_datetime(df['last_updated'], errors='coerce')
rules['last_updated_before_transaction'] = last_updated < transaction_date

pd.DataFrame({k:int(v.sum()) for k,v in rules.items()}, index=['affected_rows']).T

,affected_rows
transaction_id_required,0
amount_non_negative,1
transaction_date_valid,1
payment_method_invalid,1
currency_invalid,1
region_inconsistent,1
customer_id_missing,1
last_updated_before_transaction,1


## Task 5 - Audit summary


In [11]:
audit_summary = pd.DataFrame([
    ['Missing customer_id', int(df['customer_id'].isna().sum()), 'High', 'Impute or remove records with missing customer_id'],

    ['Duplicate transaction_id', int(df.duplicated(subset=['transaction_id']).sum()), 'Medium', 'Remove duplicate records based on transaction_id'],

    ['Invalid transaction_date', int(pd.to_datetime(df['transaction_date'], errors='coerce', format='mixed').isna().sum()), 'High', 'Correct or remove records with invalid dates'],

    ['Invalid amount (<= 0 or missing)', int((pd.to_numeric(df['amount'], errors='coerce').isna() | (pd.to_numeric(df['amount'], errors='coerce') <= 0)).sum()), 'High', 'Investigate and clean invalid transaction amounts'],

    ['Inconsistent categorical values', int((
        (df['payment_method'].astype(str) != df['payment_method'].astype(str).str.lower()) |
        (df['region'].astype(str) != df['region'].astype(str).str.upper()) |
        (~df['currency'].isin(['EUR', 'USD']))
    ).sum()), 'Medium', 'Standardize categorical values (case normalization and mapping)'],

    ['Invalid timestamp relationship', int((
        pd.to_datetime(df['last_updated'], errors='coerce') <
        pd.to_datetime(df['transaction_date'], errors='coerce', format='mixed')
    ).sum()), 'Low', 'Validate and correct timestamp inconsistencies']

], columns=['issue_type','affected_rows','severity','recommended_next_action'])

audit_summary


,issue_type,affected_rows,severity,recommended_next_action
0,Missing customer_id,1,High,Impute or remove records with missing customer_id
1,Duplicate transaction_id,1,Medium,Remove duplicate records based on transaction_id
2,Invalid transaction_date,1,High,Correct or remove records with invalid dates
3,Invalid amount (<= 0 or missing),3,High,Investigate and clean invalid transaction amounts
4,Inconsistent categorical values,3,Medium,Standardize categorical values (case normaliza...
5,Invalid timestamp relationship,1,Low,Validate and correct timestamp inconsistencies


## Task 6 - Recommended cleaning steps for next chapter

- Remove duplicate records based on transaction_id to avoid double counting.

- Standardize date formats by converting transaction_date and last_updated to a consistent datetime format and removing invalid dates.

- Normalize categorical fields (payment_method, currency, region) by applying case standardization and mapping to predefined valid values.

- Handle missing values: remove records with missing critical fields (e.g., customer_id) and decide on imputation for non-critical fields.

- Filter or flag invalid transaction amounts (negative or zero) for further investigation.


## Reflection questions

1. Which KPI gave the strongest signal?
2. Which issue should be escalated first?
3. What additional metadata would improve this audit?


Answer:

1. The error rate stands out the most, since a noticeable share of records contain invalid values like incorrect dates or problematic amounts.

2. Missing key fields and invalid values should be addressed first, because they directly affect whether the data can be used at all for analysis or reporting.

3. Having clearer metadata would help, for example defined formats for dates, allowed values for categories, and reference lists for things like currency or region codes.

## Bonus (recommended): write your own audit function

Create at least one helper function with clear parameters and docstring.
This demonstrates that your logic is reusable and understandable.


In [12]:
def summarize_rule_violations(rule_dictionary):
    """Summarize affected-row counts for each validation rule.

    Parameters
    ----------
    rule_dictionary : dict[str, pd.Series]
        Dictionary where keys are rule names and values are boolean masks.

    Returns
    -------
    pd.DataFrame
        Table with one row per rule and violation counts.
    """
    return pd.DataFrame({
        'rule_name': list(rule_dictionary.keys()),
        'affected_rows': [int(mask.sum()) for mask in rule_dictionary.values()]
    }).sort_values('affected_rows', ascending=False)

# Example usage with your `rules` dictionary:
summarize_rule_violations(rules)


,rule_name,affected_rows
1,amount_non_negative,1
2,transaction_date_valid,1
3,payment_method_invalid,1
4,currency_invalid,1
5,region_inconsistent,1
6,customer_id_missing,1
7,last_updated_before_transaction,1
0,transaction_id_required,0


### Explain your function parameters

In 3-5 lines, explain:
- what each parameter means,
- why you selected default values,
- how changing parameters affects KPI/audit results.


## Submission checklist

- [x] Dataset described
- [x] Issues mapped to dimensions
- [x] At least 3 KPIs calculated
- [x] At least 3 validation rules defined
- [x] Audit summary completed
- [x] Cleaning recommendations proposed (not implemented)
